# Serie I: modelo para planificar la siguiente zafra

**Borrador reproducible; revisión independiente pendiente.** Se predice azúcar recuperable por acre (`T_SPACRE`, lb/acre) para un año posterior, con variedad, localidad, etapa, corte planificado e información histórica disponible hasta el año anterior. Se comparan tres algoritmos y una referencia histórica sencilla.

La validación simula dos pronósticos de un año: entrenamiento hasta 2018 y validación en 2019; entrenamiento hasta 2019 y validación en 2020. Se elige por MAE promedio de esas validaciones. El año 2021 queda reservado para la evaluación final, sin usarlo para elegir algoritmo. Las filas de una misma combinación en un año nunca se reparten entre entrenamiento y evaluación.

`TCA`, `TRS_TON`, `POPN`, `MSTWT` y `FIBER` de la zafra que se quiere predecir no se usan como entradas. En particular, `TCA × TRS_TON ≈ T_SPACRE`: usarlas daría una precisión poco representativa de la planificación. Los promedios históricos del objetivo se calculan con años estrictamente anteriores; el procesamiento y la imputación se ajustan solo con el entrenamiento de cada corte temporal.

In [1]:
import sys
print("Python de ejecución:", sys.executable)
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RUTA = Path("../data/processed/serie1_etl_provisional.csv")
if not RUTA.exists():
    RUTA = Path("data/processed/serie1_etl_provisional.csv")
datos = pd.read_csv(RUTA, low_memory=False)
excluir = datos["REQUIERE_REVISION"] | datos["T_SPACRE"].isna()
excluidas = datos.loc[excluir].copy()
excluidas["MOTIVO_EXCLUSION_ANALISIS"] = np.where(
    excluidas["T_SPACRE"].isna(), "Falta variable objetivo.", "Alerta de calidad; sensibilidad con inclusión disponible."
)
excluidas.to_csv(RUTA.parent / "serie1_modelado_exclusiones.csv", index=False)
base = datos.loc[~excluir].copy()
print(f"Filas fuente analítica: {len(datos):,}; modelado: {len(base):,}; excluidas temporalmente: {len(excluidas):,}.")
print(f"Ceros fuente en el objetivo conservados: {base['T_SPACRE'].eq(0).sum()}.")

def agregar_historia(tabla):
    """Añade medias del objetivo de años previos, disponibles antes de la zafra."""
    resultado = tabla.copy()
    grupos = [(["CLONE"], "HIST_CLONE"), (["LOC"], "HIST_LOC"), (["CLONE", "CROP"], "HIST_CLONE_CROP")]
    for anio in sorted(tabla["YEAR"].unique()):
        pasado = tabla.loc[tabla["YEAR"].lt(anio)]
        actuales = tabla.loc[tabla["YEAR"].eq(anio)]
        for claves, nombre in grupos:
            historia = pasado.groupby(claves, as_index=False)["T_SPACRE"].mean().rename(columns={"T_SPACRE": nombre})
            valores = actuales[claves].merge(historia, on=claves, how="left", validate="many_to_one")[nombre]
            resultado.loc[actuales.index, nombre] = valores.to_numpy()
    return resultado

modelado = agregar_historia(base)
categoricas = ["CLONE", "LOC", "STAGE"]
modelado[categoricas] = modelado[categoricas].astype(str)
numericas = ["YEAR", "CROP", "HIST_CLONE", "HIST_LOC", "HIST_CLONE_CROP"]
entradas = categoricas + numericas
preparacion = ColumnTransformer([
    ("categorias", OneHotEncoder(handle_unknown="ignore", min_frequency=5), categoricas),
    ("numeros", Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())]), numericas),
])
algoritmos = {
    "Ridge interpretable": Ridge(alpha=10, solver="lsqr"),
    "Árbol de decisión": DecisionTreeRegressor(max_depth=6, min_samples_leaf=20, random_state=42),
    "Bosque aleatorio": RandomForestRegressor(n_estimators=120, max_depth=18, min_samples_leaf=5, max_features=0.8, n_jobs=-1, random_state=42),
}
print("Variables disponibles antes de la zafra:", entradas)

def referencia_reciente(entrenamiento, evaluacion):
    """Pronostica con el mismo clon y ambiente del año anterior, con respaldo histórico."""
    claves = ["CLONE", "LOC", "STAGE", "CROP"]
    reciente = entrenamiento.loc[entrenamiento["YEAR"].eq(entrenamiento["YEAR"].max())]
    prediccion = evaluacion[claves].join(reciente.groupby(claves)["T_SPACRE"].mean().rename("PRED"), on=claves)["PRED"]
    historica = evaluacion[claves].join(entrenamiento.groupby(claves)["T_SPACRE"].mean().rename("PRED"), on=claves)["PRED"]
    por_corte = evaluacion[["STAGE", "CROP"]].join(entrenamiento.groupby(["STAGE", "CROP"])["T_SPACRE"].mean().rename("PRED"), on=["STAGE", "CROP"])["PRED"]
    return prediccion.fillna(historica).fillna(por_corte).fillna(entrenamiento["T_SPACRE"].mean())


Python de ejecución: C:\Users\user\Desktop\Cursos\cesar\otros\yo\Prueba Santa Ana\.venv\Scripts\python.exe


Filas fuente analítica: 29,403; modelado: 28,981; excluidas temporalmente: 422.
Ceros fuente en el objetivo conservados: 8.


Variables disponibles antes de la zafra: ['CLONE', 'LOC', 'STAGE', 'YEAR', 'CROP', 'HIST_CLONE', 'HIST_LOC', 'HIST_CLONE_CROP']


## Comparación temporal y elección

Se mantienen hiperparámetros sencillos y fijos; no se hace una búsqueda extensa. Ridge representa una relación aditiva regularizada; el árbol produce reglas; el bosque permite interacciones y relaciones no lineales. La referencia predice con la media histórica de clon/corte; si no existe, usa clon, localidad y finalmente la media de entrenamiento.

MAE y RMSE se expresan en lb de azúcar/acre. Se reportan ambas: MAE para el error típico de planificación y RMSE para penalizar errores grandes. R² complementa la lectura sin sustituir la comparación contra la referencia.

Se añade una segunda referencia más exigente: el mismo clon, localidad, etapa y corte del año anterior. Si no hay esa observación, usa la historia disponible de esa combinación, luego etapa/corte y finalmente el promedio de entrenamiento.

In [2]:
resultados = []
for anio in [2019, 2020]:
    entrenamiento = modelado.loc[modelado["YEAR"].lt(anio)]
    evaluacion = modelado.loc[modelado["YEAR"].eq(anio)]
    referencia = evaluacion["HIST_CLONE_CROP"].fillna(evaluacion["HIST_CLONE"]).fillna(evaluacion["HIST_LOC"]).fillna(entrenamiento["T_SPACRE"].mean())
    predicciones = {"Referencia histórica": referencia.to_numpy(), "Referencia clon-ambiente reciente": referencia_reciente(entrenamiento, evaluacion).to_numpy()}
    for nombre, algoritmo in algoritmos.items():
        modelo = Pipeline([("preparar", clone(preparacion)), ("regresor", clone(algoritmo))])
        modelo.fit(entrenamiento[entradas], entrenamiento["T_SPACRE"])
        predicciones[nombre] = modelo.predict(evaluacion[entradas])
    for nombre, prediccion in predicciones.items():
        resultados.append({
            "ANIO_VALIDACION": anio, "MODELO": nombre,
            "MAE": mean_absolute_error(evaluacion["T_SPACRE"], prediccion),
            "RMSE": np.sqrt(mean_squared_error(evaluacion["T_SPACRE"], prediccion)),
            "R2": r2_score(evaluacion["T_SPACRE"], prediccion),
            "N": len(evaluacion),
        })
validacion = pd.DataFrame(resultados)
display(validacion.round(2))
comparacion = validacion.groupby("MODELO")[["MAE", "RMSE", "R2"]].mean().sort_values("MAE")
display(comparacion.round(2))
elegido = comparacion.loc[comparacion.index.isin(algoritmos)].index[0]
print("Algoritmo elegido por validación 2019–2020:", elegido)
validacion.to_csv(RUTA.parent / "serie1_validacion_modelos.csv", index=False)

,ANIO_VALIDACION,MODELO,MAE,RMSE,R2,N
0,2019,Referencia histórica,3088.23,3815.64,-1.59,2304
1,2019,Referencia clon-ambiente reciente,2699.06,3341.71,-0.99,2304
2,2019,Ridge interpretable,2515.28,3116.38,-0.73,2304
3,2019,Árbol de decisión,2602.75,3188.76,-0.81,2304
4,2019,Bosque aleatorio,2743.86,3344.82,-0.99,2304
5,2020,Referencia histórica,2386.03,3209.40,0.02,2207
6,2020,Referencia clon-ambiente reciente,2752.09,3502.02,-0.16,2207
7,2020,Ridge interpretable,2296.01,3076.49,0.10,2207
8,2020,Árbol de decisión,2973.15,3789.16,-0.36,2207
9,2020,Bosque aleatorio,3021.11,3795.28,-0.36,2207


,MAE,RMSE,R2
MODELO,,,
Ridge interpretable,2405.64,3096.43,-0.31
Referencia clon-ambiente reciente,2725.58,3421.87,-0.58
Referencia histórica,2737.13,3512.52,-0.78
Árbol de decisión,2787.95,3488.96,-0.59
Bosque aleatorio,2882.48,3570.05,-0.68


Algoritmo elegido por validación 2019–2020: Ridge interpretable


## Evaluación reservada: zafra 2021

Se entrena hasta 2020 y se evalúa en 2021. Se conserva la elección previa aunque otro algoritmo resulte mejor en este año; no se utiliza 2021 para retocar hiperparámetros. Además del resultado global se informa el error en etapa 5 y en clones vistos/no vistos. La validación representa el siguiente año de los ambientes del archivo, no una finca nueva sin historia.

In [3]:
entrenamiento = modelado.loc[modelado["YEAR"].le(2020)]
prueba = modelado.loc[modelado["YEAR"].eq(2021)].copy()
modelos_2021 = {}
prueba["PRED_REFERENCIA"] = prueba["HIST_CLONE_CROP"].fillna(prueba["HIST_CLONE"]).fillna(prueba["HIST_LOC"]).fillna(entrenamiento["T_SPACRE"].mean())
evaluacion_final = []
for nombre, algoritmo in algoritmos.items():
    modelo = Pipeline([("preparar", clone(preparacion)), ("regresor", clone(algoritmo))])
    modelo.fit(entrenamiento[entradas], entrenamiento["T_SPACRE"])
    modelos_2021[nombre] = modelo
    prediccion = modelo.predict(prueba[entradas])
    evaluacion_final.append({"MODELO": nombre, "MAE": mean_absolute_error(prueba["T_SPACRE"], prediccion), "RMSE": np.sqrt(mean_squared_error(prueba["T_SPACRE"], prediccion)), "R2": r2_score(prueba["T_SPACRE"], prediccion)})
evaluacion_final.append({"MODELO": "Referencia histórica", "MAE": mean_absolute_error(prueba["T_SPACRE"], prueba["PRED_REFERENCIA"]), "RMSE": np.sqrt(mean_squared_error(prueba["T_SPACRE"], prueba["PRED_REFERENCIA"])), "R2": r2_score(prueba["T_SPACRE"], prueba["PRED_REFERENCIA"])})
prediccion_reciente = referencia_reciente(entrenamiento, prueba)
evaluacion_final.append({"MODELO": "Referencia clon-ambiente reciente", "MAE": mean_absolute_error(prueba["T_SPACRE"], prediccion_reciente), "RMSE": np.sqrt(mean_squared_error(prueba["T_SPACRE"], prediccion_reciente)), "R2": r2_score(prueba["T_SPACRE"], prediccion_reciente)})
evaluacion_final = pd.DataFrame(evaluacion_final)
display(evaluacion_final.round(2))
prueba["PREDICCION"] = modelos_2021[elegido].predict(prueba[entradas])
prueba["ERROR_ABSOLUTO"] = (prueba["PREDICCION"] - prueba["T_SPACRE"]).abs()
prueba["CLON_CON_HISTORIA"] = prueba["CLONE"].isin(entrenamiento["CLONE"])
print("Error del modelo elegido por disponibilidad de historia del clon:")
display(prueba.groupby("CLON_CON_HISTORIA").agg(n=("ERROR_ABSOLUTO", "size"), mae=("ERROR_ABSOLUTO", "mean")).round(2))
print("Error por ciclo y etapa:")
display(prueba.groupby(["STAGE", "CROP"]).agg(n=("ERROR_ABSOLUTO", "size"), mae=("ERROR_ABSOLUTO", "mean")).round(2))
print("MAE en etapa 5:", round(prueba.loc[prueba["STAGE"].eq("5"), "ERROR_ABSOLUTO"].mean(), 2))
prueba[["FILA_CSV", "CLONE", "LOC", "YEAR", "STAGE", "CROP", "T_SPACRE", "PREDICCION", "PRED_REFERENCIA", "ERROR_ABSOLUTO", "CLON_CON_HISTORIA"]].to_csv(RUTA.parent / "serie1_predicciones_2021.csv", index=False)
evaluacion_final.to_csv(RUTA.parent / "serie1_evaluacion_2021.csv", index=False)

# Sensibilidad del entrenamiento: mismas entradas, algoritmo y prueba limpia.
con_alertas = agregar_historia(datos.loc[datos["T_SPACRE"].notna()])
con_alertas[categoricas] = con_alertas[categoricas].astype(str)
modelo_sensible = Pipeline([("preparar", clone(preparacion)), ("regresor", clone(algoritmos[elegido]))])
antes_2021 = con_alertas.loc[con_alertas["YEAR"].le(2020)]
modelo_sensible.fit(antes_2021[entradas], antes_2021["T_SPACRE"])
# Se comparan en exactamente las mismas filas y con historia de la alternativa.
misma_prueba = con_alertas.loc[con_alertas["FILA_CSV"].isin(prueba["FILA_CSV"])].set_index("FILA_CSV").loc[prueba["FILA_CSV"]]
print("MAE con alertas incluidas en entrenamiento e historia, sobre las mismas filas de prueba:", round(mean_absolute_error(misma_prueba["T_SPACRE"], modelo_sensible.predict(misma_prueba[entradas])), 2))
print("MAE del escenario principal, sin ceros en la prueba:", round(prueba.loc[prueba["T_SPACRE"].gt(0), "ERROR_ABSOLUTO"].mean(), 2))

,MODELO,MAE,RMSE,R2
0,Ridge interpretable,2144.55,2770.97,0.25
1,Árbol de decisión,2330.43,2983.90,0.13
2,Bosque aleatorio,2261.69,2877.42,0.19
3,Referencia histórica,2662.69,3385.62,-0.12
4,Referencia clon-ambiente reciente,2270.33,2938.43,0.16


Error del modelo elegido por disponibilidad de historia del clon:


,n,mae
CLON_CON_HISTORIA,,
False,308,2173.05
True,1980,2140.11


Error por ciclo y etapa:


n      mae
STAGE CROP              
3     0     449  2839.22
      1     188  2457.20
      2      84  1779.53
      3      28  2672.95
4     0      76  1729.03
      1      48  2675.20
5     0     382  1800.81
      1     420  1945.58
      2     335  2088.91
      3     248  1717.61
      4      30  1833.80

MAE en etapa 5: 1898.11


MAE con alertas incluidas en entrenamiento e historia, sobre las mismas filas de prueba: 2135.1
MAE del escenario principal, sin ceros en la prueba: 2144.55


## Interpretabilidad y uso gerencial

El árbol muestra reglas de planificación sobre las entradas disponibles antes de cosechar. Ridge permite explicar un efecto aditivo condicionado a las otras variables; sus coeficientes regularizados no deben confundirse con efectos causales. El algoritmo con menor MAE temporal se propone para planificación, condicionado a que mejore a la referencia histórica y a su desempeño en etapa 5.

Una predicción puntual sirve para comparar escenarios y presupuestar producción esperada; no asegura el resultado de una parcela. El error observado en 2021 debe acompañar la cifra y desglosarse por ambiente. Para obtener intervalos de predicción con cobertura comprobada hacen falta calibración temporal adicional y revisión específica; MAE por sí solo no es un intervalo de confianza.

In [4]:
arbol = modelos_2021["Árbol de decisión"]
nombres = arbol.named_steps["preparar"].get_feature_names_out().tolist()
print(export_text(arbol.named_steps["regresor"], feature_names=nombres, max_depth=3))
mae_referencia = evaluacion_final.loc[evaluacion_final["MODELO"].eq("Referencia histórica"), "MAE"].iloc[0]
mae_elegido = evaluacion_final.loc[evaluacion_final["MODELO"].eq(elegido), "MAE"].iloc[0]
print(f"Algoritmo elegido: {elegido}; MAE 2021: {mae_elegido:.2f} lb/acre; mejora frente a referencia: {100 * (1 - mae_elegido / mae_referencia):.1f} %.")
print("Ejecución reproducida; comprobación independiente de métricas y supuestos pendiente.")
ridge = modelos_2021["Ridge interpretable"]
preparador = ridge.named_steps["preparar"]
coeficientes = ridge.named_steps["regresor"].coef_
nombres = preparador.get_feature_names_out()
tabla_coeficientes = pd.DataFrame({"VARIABLE": nombres, "COEFICIENTE": coeficientes})
print("Coeficientes numéricos de Ridge por unidad original de entrada:")
coef_numericos = tabla_coeficientes.loc[tabla_coeficientes["VARIABLE"].str.startswith("numeros__")].copy()
coef_numericos["COEFICIENTE_POR_UNIDAD"] = coef_numericos["COEFICIENTE"].to_numpy() / preparador.named_transformers_["numeros"].named_steps["escalar"].scale_
display(coef_numericos[["VARIABLE", "COEFICIENTE_POR_UNIDAD"]].round(3))
print("Son asociaciones condicionadas y regularizadas; no representan efectos causales de cambiar año o corte.")
mae_reciente = evaluacion_final.loc[evaluacion_final["MODELO"].eq("Referencia clon-ambiente reciente"), "MAE"].iloc[0]
print(f"Mejora frente a referencia clon-ambiente reciente en 2021: {100 * (1 - mae_elegido / mae_reciente):.1f} %.")
print(f"Error absoluto agregado / azúcar observada en 2021: {100 * prueba['ERROR_ABSOLUTO'].sum() / prueba['T_SPACRE'].sum():.1f} %.")
print(f"Equivalente del MAE en ingreso bruto a US$0.20/lb: US${mae_elegido * 0.20:.2f}/acre; no es un intervalo de confianza ni el error del aporte económico total.")


|--- numeros__HIST_LOC <= 0.11
|   |--- numeros__CROP <= 0.55
|   |   |--- categorias__LOC_AFL <= 0.50
|   |   |   |--- numeros__HIST_CLONE_CROP <= 1.24
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- numeros__HIST_CLONE_CROP >  1.24
|   |   |   |   |--- truncated branch of depth 3
|   |   |--- categorias__LOC_AFL >  0.50
|   |   |   |--- categorias__STAGE_2 <= 0.50
|   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- categorias__STAGE_2 >  0.50
|   |   |   |   |--- truncated branch of depth 3
|   |--- numeros__CROP >  0.55
|   |   |--- numeros__HIST_LOC <= -0.60
|   |   |   |--- categorias__LOC_MAGNOLIA <= 0.50
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- categorias__LOC_MAGNOLIA >  0.50
|   |   |   |   |--- truncated branch of depth 3
|   |   |--- numeros__HIST_LOC >  -0.60
|   |   |   |--- categorias__LOC_AFL <= 0.50
|   |   |   |   |--- truncated branch of depth 3
|   |   |   |--- categorias__LOC_AFL >  0.50
|   |   |   |   |--- v

,VARIABLE,COEFICIENTE_POR_UNIDAD
631,numeros__YEAR,-25.069
632,numeros__CROP,-801.670
633,numeros__HIST_CLONE,0.013
634,numeros__HIST_LOC,-0.216
635,numeros__HIST_CLONE_CROP,0.058


Son asociaciones condicionadas y regularizadas; no representan efectos causales de cambiar año o corte.
Mejora frente a referencia clon-ambiente reciente en 2021: 5.5 %.
Error absoluto agregado / azúcar observada en 2021: 25.5 %.
Equivalente del MAE en ingreso bruto a US$0.20/lb: US$428.91/acre; no es un intervalo de confianza ni el error del aporte económico total.


## Modelo para la siguiente zafra disponible

Después de conservar la evaluación de 2021, se reajusta el algoritmo elegido con todos los datos disponibles hasta 2021. La siguiente entrada ilustra el uso para 2022, una zafra posterior a la última del archivo. No es un pronóstico de una zafra actual ni una observación evaluada: su objetivo es mostrar cómo generar entradas e historia disponibles antes de cosechar.

In [5]:
modelo_planificacion = Pipeline([("preparar", clone(preparacion)), ("regresor", clone(algoritmos[elegido]))])
modelo_planificacion.fit(modelado[entradas], modelado["T_SPACRE"])
localidad_ejemplo = base.loc[base["STAGE"].eq(5), "LOC"].mode().iloc[0]
entrada = pd.DataFrame([{"CLONE": "HOCP14-885", "LOC": localidad_ejemplo, "STAGE": 5, "YEAR": 2022, "CROP": 2, "T_SPACRE": np.nan}])
con_futuro = pd.concat([base, entrada], ignore_index=True)
ejemplo = agregar_historia(con_futuro).tail(1).copy()
ejemplo[categoricas] = ejemplo[categoricas].astype(str)
ejemplo["PREDICCION_LB_ACRE"] = modelo_planificacion.predict(ejemplo[entradas])
display(ejemplo[["CLONE", "LOC", "YEAR", "STAGE", "CROP", "PREDICCION_LB_ACRE"]].round(2))
print("Predicción ilustrativa; su resultado real no está en el archivo.")

,CLONE,LOC,YEAR,STAGE,CROP,PREDICCION_LB_ACRE
28981,HOCP14-885,RHEBERT,2022,5,2,9129.8


Predicción ilustrativa; su resultado real no está en el archivo.
